# Day 24 — pytest: **Fixtures & Parametrize**

**Phase 1 · Module 3: Prompt Engineering & RAG** · ~30 min

You can't ship a RAG agent to Barclays on *"it worked when I ran it."* You need tests that pin behaviour — but tests that call a real LLM are slow, costly, and flaky. Two pytest features fix that:

1. **Fixtures** — set up a **mocked** LLM client once, inject it into every test.
2. **`@pytest.mark.parametrize`** — run one test body across many query types with no copy-paste.
3. **`conftest.py`** — share fixtures across all test files.

We write real test files to a temp dir and run `pytest` on them from the notebook, so you see actual pass/fail output — no keys, no real LLM call.

> Needs `pytest` (`pip install pytest`). Tests run via subprocess against a throwaway dir.

## 1. A tiny RAG agent to test

The system under test: retrieve a context chunk, then answer. In tests we never call a real model — we inject a **fake** one. So the agent must take its LLM client as a dependency (dependency injection), not hard-code it.

In [1]:
import os, tempfile, subprocess, sys

WORK = tempfile.mkdtemp(prefix="pytest_demo_")
print("workdir:", WORK)

workdir: /var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/pytest_demo_oqw8yxyy


In [2]:
src = """
FAQ = {
    "isa allowance": "The annual ISA allowance is 20000 pounds.",
    "overdraft":     "Arranged overdraft interest is charged daily.",
    "contactless":   "Contactless is capped at 100 pounds per tap.",
}

def retrieve(query):
    for key, doc in FAQ.items():
        if any(w in query.lower() for w in key.split()):
            return doc
    return ""

def answer(query, llm):
    ctx = retrieve(query)
    if not ctx:
        return "I do not have that information."
    return llm.complete(f"Context: {ctx} || Q: {query} || A:")
"""
open(os.path.join(WORK, "rag_agent.py"), "w").write(src)
print("wrote rag_agent.py")

wrote rag_agent.py


☝ `answer(query, llm)` takes the LLM as an argument — that seam is what lets a test swap in a fake. Hard-coding `Bedrock()` inside the function would force every test to hit the network. **Injectable dependencies are testable dependencies.**

## 2. `conftest.py` — a shared, session-scoped mock LLM

A **fixture** builds a test dependency; `scope="session"` builds it **once** for the whole run (cheap here, essential when the fixture is expensive). Put it in `conftest.py` and pytest auto-injects it into any test that names it as an argument — no import needed.

In [3]:
src = """
import pytest

# FakeLLM: deterministic stand-in for a real Bedrock/Claude client.
class FakeLLM:
    def __init__(self):
        self.calls = 0
    def complete(self, prompt):
        self.calls += 1
        # echo the retrieved context back as the answer -> deterministic, keyless
        return prompt.split("Context: ")[1].split(" || Q:")[0]

@pytest.fixture(scope="session")
def llm():
    return FakeLLM()
"""
open(os.path.join(WORK, "conftest.py"), "w").write(src)
print("wrote conftest.py")

wrote conftest.py


☝ `@pytest.fixture(scope="session")` + placing it in `conftest.py` = one `FakeLLM`, shared across every test file in the dir, injected by name. Swap `FakeLLM` for a `unittest.mock.MagicMock` when you want to *assert on calls*; use a deterministic fake like this when you want to *assert on output*.

## 3. `@pytest.mark.parametrize` — one test, many query types

The Day-24 brief: test RAG across **5 query types**. Without parametrize you'd copy the test body 5 times. With it, one body runs once per tuple, each as its own reported test case — a failure names the exact input that broke.

In [4]:
src = """
import pytest
from rag_agent import answer

@pytest.mark.parametrize("query, expected_substr", [
    ("what is the isa allowance",        "20000"),
    ("overdraft interest",               "daily"),
    ("contactless limit",                "100 pounds"),
    ("ISA ALLOWANCE in caps",            "20000"),       # case-insensitive
    ("weather tomorrow",                 "do not have"), # out-of-domain -> refusal
])
def test_answer_contains(query, expected_substr, llm):    # llm injected from conftest
    assert expected_substr in answer(query, llm)

def test_fixture_is_shared(llm):
    assert llm.calls >= 0   # same instance across tests
"""
open(os.path.join(WORK, "test_rag.py"), "w").write(src)
print("wrote test_rag.py")

wrote test_rag.py


☝ Five `(query, expected)` tuples → five independent test cases from one function; the `llm` argument is filled by the fixture. The out-of-domain case pins the **refusal** path — a RAG agent that hallucinates on unknown queries is the failure mode Barclays cares about most.

## 4. Run pytest and read the report

In [5]:
res = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--no-header", "-p", "no:cacheprovider"],
    cwd=WORK, capture_output=True, text=True,
)
print(res.stdout[-1600:] if res.stdout else res.stderr[-1600:])
print("exit code:", res.returncode, "(0 = all passed)")

============================= test session starts ==============================
collecting ... collected 6 items

test_rag.py::test_answer_contains[what is the isa allowance-20000] PASSED [ 16%]
test_rag.py::test_answer_contains[overdraft interest-daily] PASSED       [ 33%]
test_rag.py::test_answer_contains[contactless limit-100 pounds] PASSED   [ 50%]
test_rag.py::test_answer_contains[ISA ALLOWANCE in caps-20000] PASSED    [ 66%]
test_rag.py::test_answer_contains[weather tomorrow-do not have] PASSED   [ 83%]
test_rag.py::test_fixture_is_shared PASSED                               [100%]

============================== 6 passed in 0.00s ===============================

exit code: 0 (0 = all passed)


☝ Each parametrized case reports on its own line (`test_answer_contains[what is the isa allowance-20000] PASSED`) — so a regression tells you *which query* broke, not just that "a test failed." That granularity is why parametrize beats a `for`-loop inside one test.

## 5. Recap — fast, deterministic agent tests

| Feature | Call | Buys you |
|---|---|---|
| **Fixture** | `@pytest.fixture(scope="session")` | build the mock LLM once, inject everywhere |
| **conftest.py** | drop fixtures in it | shared across all test files, no imports |
| **Parametrize** | `@pytest.mark.parametrize` | one body × many inputs, each reported separately |
| **Dependency injection** | `answer(query, llm)` | swap real client for a fake — no network in tests |
| **Refusal case** | out-of-domain tuple | pin the no-hallucination contract |
